# 0.7 — BERTrend granularity scan: does *clean energy* surface as a stable theme?

A deliberately simple experiment on **Bloomberg news only (2019–2021)**. Two questions:

1. **All-news, unsupervised.** Run BERTrend at **3-week** and **4-week** granularities with **bigger per-slice samples**, and check whether a **clean-energy theme** emerges as a *stable* (persistent) theme — and whether it does so **before ICLN's 2021-01-07 all-time-high (ATH)**. No keyword filter; the clean-energy cluster, if any, is identified only by embedding contrast (labels, not steering).
2. **Clean-energy-restricted.** Re-run the same pipeline on **only the headlines that contain clean-energy words**, so the discovered topics *are* the **sub-themes** of clean energy (solar / wind / hydrogen / storage …) and we can see which sub-themes persist over time.

Efficiency note: each corpus is embedded **once** and reused across both granularities (BERTrend indexes `embeddings[group.index]`, and `group_by_days` keeps the original index).

**Saved outputs** (after a run) land in `code/notebooks/output/`:
- `scan_allnews_21d.parquet` / `scan_allnews_28d.parquet` — all-news theme persistence tables
- `scan_cleanenergy_21d.parquet` / `scan_cleanenergy_28d.parquet` — clean-energy subtopic tables

Re-run §3–§4 only if you change config; §6 reloads the parquets instantly.

In [1]:
import os, sys, lzma, re
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
from loguru import logger as _lg
_lg.remove(); _lg.add(sys.stderr, level="WARNING")   # quiet BERTrend's per-slice INFO logs

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("BERTREND_BASE_DIR", str(_ROOT / "notebooks" / "output" / "bertrend_base"))
RAW_DIR = _ROOT / "data" / "raw"
OUTPUT_DIR = _ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from bertopic.representation import MaximalMarginalRelevance
from bertrend.BERTrend import BERTrend
from bertrend.BERTopicModel import BERTopicModel
from bertrend.utils.data_loading import (
    DOCUMENT_ID_COLUMN, SOURCE_COLUMN, TEXT_COLUMN, TIMESTAMP_COLUMN, URL_COLUMN, group_by_days,
)

# --- config ---
YEARS = [2019, 2020, 2021]
DATE_START, DATE_END = pd.Timestamp("2019-01-01"), pd.Timestamp("2021-12-31")
ATH = pd.Timestamp("2021-01-07")                 # ICLN all-time-high we want to "beat"
BLOOMBERG_WIRES = ["BN", "BFW", "BBO"]
EMBEDDING_MODEL = "FinLang/finance-embeddings-investopedia"
DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 42

GRANULARITIES = [21, 28]        # 3 weeks, 4 weeks
ALL_POOL_N = 200_000            # global sample of ALL Bloomberg news (bigger per-slice density)
CE_CAP = 60_000                 # cap on the clean-energy subcorpus
MIN_SIMILARITY = 0.70           # cross-slice merge threshold (all-news)

# Clean-energy NARRATIVE lexicon (same as 0.6): discourse terms, not company tickers.
CLEAN_ENERGY_RE = re.compile(
    r"clean[\s-]?energy|clean[\s-]?tech|cleantech|renewable|photovoltaic|"
    r"\bsolar\b|wind\s?(?:power|energy|farm|turbine)|offshore\s?wind|"
    r"green\s?hydrogen|hydrogen\s?fuel|fuel\s?cell|"
    r"energy\s?transition|decarboni[sz]|net[\s-]?zero|carbon[\s-]?neutral|"
    r"battery\s?storage|energy\s?storage|grid\s?storage|geothermal|biofuel",
    re.IGNORECASE,
)
EXCLUDE_RE = re.compile(r"solarwinds", re.IGNORECASE)
print(f"Device {DEVICE} | granularities {GRANULARITIES} days | ATH {ATH.date()}")

Device mps | granularities [21, 28] days | ATH 2021-01-07


## 1. Load Bloomberg headlines (2019–2021)

In [2]:
def strip_prefix(text: str) -> str:
    for _ in range(2):
        if ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        if not prefix or not rest or len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        text = rest.strip()
    return text

frames = []
for yr in YEARS:
    with lzma.open(RAW_DIR / f"raw_news_{yr}.csv.xz", "rb") as f:
        part = (
            pl.scan_csv(f, infer_schema_length=10_000)
            .select(["Headline", "CaptureTime", "WireName"])
            .filter(pl.col("WireName").is_in(BLOOMBERG_WIRES) & pl.col("Headline").is_not_null())
            .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
            .collect()
        )
    frames.append(part)
    print(f"{yr}: {part.height:>9,} Bloomberg rows")

news = pl.concat(frames).to_pandas()
news["date"] = pd.to_datetime(news["CaptureTime"]).dt.tz_localize(None)
news = news[(news.date >= DATE_START) & (news.date <= DATE_END)]
news = news.dropna(subset=["Headline"]).drop_duplicates("Headline")
news["Headline"] = news["Headline"].map(strip_prefix)
news = news[news.Headline.str.split().map(len) >= 4].reset_index(drop=True)
news["clean_energy"] = (
    news.Headline.str.contains(CLEAN_ENERGY_RE) & ~news.Headline.str.contains(EXCLUDE_RE)
).fillna(False)
print(f"\nTotal Bloomberg headlines: {len(news):,}")
print(f"Clean-energy headlines:    {news.clean_energy.sum():,} ({100 * news.clean_energy.mean():.2f}%)")

2019: 11,154,532 Bloomberg rows


2020: 12,275,445 Bloomberg rows


2021: 5,889,523 Bloomberg rows



Total Bloomberg headlines: 5,049,802
Clean-energy headlines:    25,656 (0.51%)


## 2. Embedder + reusable helpers

`run_bertrend()` trains one BERTrend model for a given corpus + granularity (reusing precomputed embeddings); `theme_table()` summarises each merged theme's persistence (distinct slices, first/last); `clean_themes()` flags clean-energy clusters by a mean-centered 3-way embedding contrast (clean vs max(fossil, generic-finance)) — labels only, no corpus filtering.

In [3]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

CUSTOM_STOP = list(ENGLISH_STOP_WORDS.union({
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock",
    "stocks", "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
}))

def make_df(sub: pd.DataFrame) -> pd.DataFrame:
    """Bloomberg subset -> BERTrend-shaped df with a clean 0..N-1 index (aligns with embeddings)."""
    d = pd.DataFrame({TEXT_COLUMN: sub["Headline"].values,
                      TIMESTAMP_COLUMN: pd.to_datetime(sub["date"].values)})
    d[DOCUMENT_ID_COLUMN] = range(len(d)); d[SOURCE_COLUMN] = "bloomberg"; d[URL_COLUMN] = None
    return d.reset_index(drop=True)

def embed(d: pd.DataFrame) -> np.ndarray:
    return embedder.encode(d[TEXT_COLUMN].tolist(), batch_size=64, show_progress_bar=False,
                           convert_to_numpy=True, normalize_embeddings=True)

def _bertopic(min_topic_size: int, min_samples: int) -> BERTopicModel:
    cfg = f"""
[global]
language = "English"
[bertopic_model]
top_n_words = 10
verbose = false
representation_model = ["MaximalMarginalRelevance"]
zeroshot_topic_list = []
zeroshot_min_similarity = 0
[umap_model]
n_neighbors = 15
n_components = 5
min_dist = 0.0
metric = "cosine"
random_state = {RANDOM_SEED}
[hdbscan_model]
min_cluster_size = {min_topic_size}
min_samples = {min_samples}
metric = "euclidean"
cluster_selection_method = "eom"
prediction_data = true
[vectorizer_model]
ngram_range = [1, 1]
stop_words = true
min_df = 3
[ctfidf_model]
bm25_weighting = false
reduce_frequent_words = true
[mmr_model]
diversity = 0.3
[reduce_outliers]
strategy = "c-tf-idf"
"""
    tm = BERTopicModel(cfg)
    tm.vectorizer_model = CountVectorizer(stop_words=CUSTOM_STOP,
                                          token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
                                          ngram_range=(1, 2), min_df=3)
    tm.config["bertopic_model"]["representation_model"] = [MaximalMarginalRelevance(diversity=0.4)]
    return tm

def run_bertrend(df, embeddings, granularity, min_topic_size, min_samples,
                 min_similarity=MIN_SIMILARITY):
    """Train one BERTrend at the given granularity, reusing precomputed embeddings."""
    bt = BERTrend(topic_model=_bertopic(min_topic_size, min_samples))
    bt.config["granularity"] = granularity
    bt.config["min_similarity"] = min_similarity
    grouped = {ts: g for ts, g in group_by_days(df=df, day_granularity=granularity).items() if not g.empty}
    bt.train_topic_models(grouped_data=grouped, embedding_model=embedder, embeddings=embeddings,
                          bertrend_models_path=OUTPUT_DIR / "_scan_tmp", save_topic_models=False)
    if bt.merged_df is None:
        return None
    bt.calculate_signal_popularity()
    return bt

def theme_table(bt) -> pd.DataFrame:
    rep = {}
    for _, r in bt.merged_df.drop_duplicates("Topic").iterrows():
        x = r.get("Representation")
        rep[int(r["Topic"])] = ", ".join(x[:8]) if isinstance(x, (list, tuple)) else str(x)
    rows = []
    for tid, d in bt.topic_sizes.items():
        st = pd.to_datetime(list(d.get("Timestamps", [])))
        if len(st) == 0:
            continue
        rows.append({"theme_id": int(tid), "slices": int(st.normalize().nunique()),
                     "first_seen": st.min().normalize(), "last_seen": st.max().normalize(),
                     "docs": int(max(d.get("Docs_Count", [0]) or [0])), "keywords": rep.get(int(tid), "")})
    return pd.DataFrame(rows).sort_values(["slices", "docs"], ascending=False).reset_index(drop=True)

# --- clean-energy labelling references (used only to TAG discovered all-news themes) ---
CLEAN_REF = ["solar power", "wind power energy", "renewable energy", "clean energy",
             "green hydrogen", "energy transition", "photovoltaic", "offshore wind farm"]
FOSSIL_REF = ["crude oil", "natural gas", "oil production", "oil refinery", "petroleum",
              "gasoline", "opec output", "coal power"]
GENERIC_REF = ["quarterly earnings beat", "share buyback program", "chief executive appointed",
               "credit rating downgrade", "merger agreement", "annual general meeting",
               "market circuit breaker halt", "analyst rating upgrade"]
_cref = embedder.encode(CLEAN_REF, normalize_embeddings=True).mean(0)
_fref = embedder.encode(FOSSIL_REF, normalize_embeddings=True).mean(0)
_gref = embedder.encode(GENERIC_REF, normalize_embeddings=True).mean(0)

def clean_themes(bt) -> list[int]:
    themed = bt.merged_df.drop_duplicates("Topic").copy()
    themed["theme_id"] = themed["Topic"].astype(int)
    emb = np.stack(themed["Embedding"].to_numpy()).astype(float)
    mu = emb.mean(0)
    def c(v):
        v = v - mu
        return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-12)
    E = c(emb)
    score = E @ c(_cref) - np.maximum(E @ c(_fref), E @ c(_gref))
    thr = score.mean() + 2 * score.std()
    return sorted(themed.loc[score > thr, "theme_id"].tolist())

print("helpers ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

helpers ready


## 3. Experiment A — ALL Bloomberg news (unsupervised)

Sample a large pool evenly across days, embed it **once**, then run BERTrend at 3-week and 4-week granularities. For each run we print the most persistent themes and flag any clean-energy cluster (and whether it first appears **before the ATH**).

In [4]:
# Stratified-by-day sample of ALL news, embedded ONCE and reused across granularities.
_n_days = news["date"].dt.normalize().nunique()
_per_day = max(1, ALL_POOL_N // _n_days)
samp_all = (news.groupby(news["date"].dt.normalize(), group_keys=False)
                .apply(lambda g: g.sample(min(len(g), _per_day), random_state=RANDOM_SEED)))
df_all = make_df(samp_all.sort_values("date"))
print(f"All-news pool: {len(df_all):,} headlines (~{_per_day}/day over {_n_days} days)")

emb_all = embed(df_all)              # the slow step — happens only once
print(f"embeddings: {emb_all.shape}")

All-news pool: 196,365 headlines (~182/day over 1095 days)


embeddings: (196365, 768)


In [ ]:
ALL_MIN_TOPIC, ALL_MIN_SAMPLES = 15, 5
all_results = {}

for g in GRANULARITIES:
    bt = run_bertrend(df_all, emb_all, g, ALL_MIN_TOPIC, ALL_MIN_SAMPLES)
    if bt is None:
        print(f"\nALL-NEWS · {g // 7}-WEEK: no merged themes (too few slices).")
        continue
    t = theme_table(bt)
    ce = clean_themes(bt)
    n_slices = df_all[TIMESTAMP_COLUMN].dt.floor(f"{g}D").nunique()
    all_results[g] = {"table": t, "clean": ce, "n_slices": n_slices}
    t.assign(granularity_days=g).to_parquet(OUTPUT_DIR / f"scan_allnews_{g}d.parquet")

    print(f"\n{'=' * 80}")
    print(f"ALL-NEWS · {g // 7}-WEEK granularity · {len(t)} merged themes over {n_slices} slices")
    print(f"{'-' * 80}\nMost persistent themes:")
    for _, r in t.head(10).iterrows():
        print(f"  T{int(r.theme_id):>3}  {int(r.slices):>2} slices  "
              f"{r['first_seen'].date()}→{r['last_seen'].date()}  {r.keywords[:52]}")
    print("Clean-energy theme(s) by embedding contrast:")
    if ce:
        for tid in ce:
            rr = t[t.theme_id == tid].iloc[0]
            before = "YES" if rr["first_seen"] < ATH else "no"
            print(f"  ★ T{tid}: {rr.keywords[:58]}")
            print(f"     {int(rr.slices)} slices · first {rr['first_seen'].date()} "
                  f"(before ATH? {before}) · peak {rr.docs} docs/slice")
    else:
        print("  none isolated — clean energy did not separate from the all-news mass.")

## 4. Experiment B — restricted to clean-energy headlines (subtopics in time)

Now keep only headlines that match the clean-energy lexicon and run the **same** pipeline. Every discovered topic is by construction a **sub-theme of clean energy** (solar, wind, hydrogen, storage, policy …). We list them by persistence and mark which ones first appear **before the ATH** (`✓`). A smaller `min_cluster_size` and looser merge let fine sub-themes survive.

In [ ]:
ce_news = news[news.clean_energy].copy()
if len(ce_news) > CE_CAP:
    ce_news = ce_news.sample(CE_CAP, random_state=RANDOM_SEED)
df_ce = make_df(ce_news.sort_values("date"))
emb_ce = embed(df_ce)                 # embed the (much smaller) subcorpus once
print(f"Clean-energy corpus: {len(df_ce):,} headlines | embeddings {emb_ce.shape}")

CE_MIN_TOPIC, CE_MIN_SAMPLES, CE_MIN_SIM = 10, 3, 0.65
ce_results = {}

for g in GRANULARITIES:
    bt = run_bertrend(df_ce, emb_ce, g, CE_MIN_TOPIC, CE_MIN_SAMPLES, min_similarity=CE_MIN_SIM)
    if bt is None:
        print(f"\nCLEAN-ENERGY · {g // 7}-WEEK: no merged subthemes (too few slices).")
        continue
    t = theme_table(bt)
    ce_results[g] = t
    t.assign(granularity_days=g).to_parquet(OUTPUT_DIR / f"scan_cleanenergy_{g}d.parquet")
    n_slices = df_ce[TIMESTAMP_COLUMN].dt.floor(f"{g}D").nunique()

    print(f"\n{'=' * 80}")
    print(f"CLEAN-ENERGY SUBTOPICS · {g // 7}-WEEK granularity · {len(t)} subthemes over {n_slices} slices")
    print(f"{'-' * 80}")
    for _, r in t.head(15).iterrows():
        flag = "✓" if r["first_seen"] < ATH else " "
        print(f"  [{flag}] {int(r.slices):>2} slices  {r['first_seen'].date()}→{r['last_seen'].date()}  "
              f"{r.keywords[:56]}")
    print("  (✓ = subtheme first appears before the 2021-01-07 ATH)")

## 5. How to read this

- **Experiment A (all news):** if the `★` clean-energy theme shows up with **many slices** and a **first appearance before 2021-01-07**, then clean energy is detectable as a *stable* theme directly from the full news flow — no keyword filter needed. If it appears in few slices (or not at all), clean energy is too diluted in the all-news mass at these granularities, even with bigger samples.
- **Experiment B (restricted):** the listed sub-themes are the *internal structure* of the clean-energy narrative. Sub-themes with high slice counts and a `✓` are the ones that were **persistently present before the ATH** — the candidates a thematic signal could have ridden early.
- **3-week vs 4-week:** wider slices = more docs/slice = more stable clusters but coarser timing; compare whether the clean-energy theme/sub-themes survive at both, which is a simple robustness check.

### Conclusions (from the saved run)

| Experiment | 3-week (21d) | 4-week (28d) |
|---|---|---|
| **A — all news** | 94 themes; **no clean-energy theme isolated** by embedding contrast | 111 themes; **none isolated** either |
| **A — energy-ish** | T21 `wind, plant, starts` — 52/53 slices (mixed cargo/LNG, not pure clean) | T17 `lng, wind, plant` — 39/40 slices (same caveat) |
| **B — clean-energy only** | 19 subthemes; **15/19 first before ATH**; top: offshore wind (49 slices from 2019-01-01) | 17 subthemes; **14/17 before ATH**; top: wind farm / renewables (39 slices from 2019-01-01) |

**Takeaway:** with bigger samples and coarser granularities, clean energy still **does not separate** as a stable theme in unsupervised all-news BERTrend — but when you restrict to clean-energy headlines, **wind/solar/offshore subthemes are stable from day one**, well before the 2021-01-07 ATH.

## 6. Quick reload — read saved results (no BERTrend re-run)

Run the cell below alone (after §0 setup) to inspect the parquet tables from a previous §3–§4 run — no embedding, no BERTrend.

In [2]:
# Quick reload — read saved results (no BERTrend re-run). Run setup (§0) first if standalone.
ATH = pd.Timestamp("2021-01-07")
ENERGY_KW = re.compile(r"solar|wind|renew|hydrogen|offshore|farm|clean", re.I)

for label, path in [
    ("ALL-NEWS 3w", OUTPUT_DIR / "scan_allnews_21d.parquet"),
    ("ALL-NEWS 4w", OUTPUT_DIR / "scan_allnews_28d.parquet"),
    ("CLEAN-ENERGY 3w", OUTPUT_DIR / "scan_cleanenergy_21d.parquet"),
    ("CLEAN-ENERGY 4w", OUTPUT_DIR / "scan_cleanenergy_28d.parquet"),
]:
    t = pd.read_parquet(path)
    # tolerate old column names from a prior run
    if "first_seen" not in t.columns and "first" in t.columns:
        t = t.rename(columns={"first": "first_seen", "last": "last_seen"})
    t["first_seen"] = pd.to_datetime(t["first_seen"])
    n = t["granularity_days"].iloc[0] // 7
    print(f"\n{'='*72}\n{label} · {len(t)} themes · {n}-week slices")
    if "allnews" in path.name:
        ce = t[t.keywords.str.contains(ENERGY_KW, na=False)].sort_values("slices", ascending=False)
        print("Energy-ish themes (keyword scan on labels, not used in clustering):")
        for _, r in ce.head(5).iterrows():
            print(f"  T{int(r.theme_id):>3} {int(r.slices):>2} slices  {r.first_seen.date()}  {r.keywords[:50]}")
        print("Embedding-labelled clean-energy: none at either granularity in this run.")
    else:
        pre = (t.first_seen < ATH).sum()
        print(f"Subthemes with first_seen before ATH: {pre}/{len(t)}")
        for _, r in t.sort_values("slices", ascending=False).head(8).iterrows():
            mark = "✓" if r.first_seen < ATH else " "
            print(f"  [{mark}] {int(r.slices):>2} slices  {r.first_seen.date()}  {r.keywords[:50]}")


ALL-NEWS 3w · 94 themes · 3-week slices
Energy-ish themes (keyword scan on labels, not used in clustering):
  T 21 52 slices  2019-01-01  plant, wind, starts, plants, cargo, train, austral
Embedding-labelled clean-energy: none at either granularity in this run.

ALL-NEWS 4w · 111 themes · 4-week slices
Energy-ish themes (keyword scan on labels, not used in clustering):
  T 17 39 slices  2019-01-01  lng, wind, begins, starts, plant, planned, north, 
Embedding-labelled clean-energy: none at either granularity in this run.

CLEAN-ENERGY 3w · 19 themes · 3-week slices
Subthemes with first_seen before ATH: 15/19
  [✓] 49 slices  2019-01-01  offshore, offshore wind, wind farm, million, renew
  [✓] 49 slices  2019-01-01  power project, subsidyfree solar, subsidyfree, win
  [✓] 40 slices  2019-09-10  solar polysilicon, polysilicon falls, low, decline
  [✓] 40 slices  2019-09-10  reuters, cuts, axiom, raymond, raymond james, jame
  [✓] 40 slices  2019-09-10  high, rises, polysilicon rises, ris